In [11]:
!cd "/content/drive/MyDrive/CUHK_Stuff/AI-for-Business-Research/Assignment/Final Project/Study2/code" && python 00_setup.py && python 01_data_loading.py

Project root: /content/drive/.shortcut-targets-by-id/1KKOoZevY1ahDrKYLLQ-dQDM6ItZ8AsDk/Final Project/Study2
Device: cuda
Seed set to: 42
Data raw directory: /content/drive/.shortcut-targets-by-id/1KKOoZevY1ahDrKYLLQ-dQDM6ItZ8AsDk/Final Project/Study2/data/raw
All output directories created.
STEP 1: Loading raw CSVs

--- transactions.csv ---
  Shape: (150000, 10)
  Columns: ['transaction_id', 'customer_id', 'product_id', 'order_date', 'order_value', 'payment_method', 'device_type', 'discount_applied', 'shipping_delay_days', 'fraud_label']
  Dtypes:
transaction_id          object
customer_id             object
product_id              object
order_date              object
order_value            float64
payment_method          object
device_type             object
discount_applied       float64
shipping_delay_days      int64
fraud_label              int64
  Missing values:
transaction_id         0
customer_id            0
product_id             0
order_date             0
order_value       

In [12]:
!cd "/content/drive/MyDrive/CUHK_Stuff/AI-for-Business-Research/Assignment/Final Project/Study2/code" && python 02_feature_engineering.py

STEP 1: Loading joined dataset
  Loaded: (150000, 28)

STEP 2: Dropping non-feature columns
  Dropped: ['transaction_id', 'customer_id', 'product_id', 'churn_label', 'behavior_churn_signal']
  Remaining columns (23): ['order_date', 'order_value', 'payment_method', 'device_type', 'discount_applied', 'shipping_delay_days', 'fraud_label', 'age', 'gender', 'country', 'registration_date', 'loyalty_score', 'lifetime_value', 'category', 'price', 'margin_percentage', 'popularity_score', 'avg_session_time', 'pages_per_session', 'cart_abandon_rate', 'return_rate', 'support_tickets', 'review_score']

STEP 3: Temporal feature extraction
  Derived features:
    order_hour: 0-0
    order_dayofweek: 0-6 (0=Mon, 6=Sun)
    order_month: 1-12
    order_is_weekend: {0: 106976, 1: 43024}
    customer_tenure_days: -1630-2360

STEP 4: Column type classification
  Numerical (16): ['order_value', 'discount_applied', 'shipping_delay_days', 'age', 'loyalty_score', 'lifetime_value', 'price', 'margin_percentage',

In [13]:
!cd "/content/drive/MyDrive/CUHK_Stuff/AI-for-Business-Research/Assignment/Final Project/Study2/code" && python 03_model_1dcnn.py

Feature count from metadata: 47
MODEL VERIFICATION

Device: cuda
Input features: 47

Model architecture:
FraudDetectionCNN1D(
  (conv_block1): ConvBlock(
    (block): Sequential(
      (0): Conv1d(1, 64, kernel_size=(3,), stride=(1,), padding=(1,))
      (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
      (4): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (residual_block): ResidualBlock(
    (conv1): Conv1d(64, 64, kernel_size=(3,), stride=(2,), padding=(1,))
    (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu1): ReLU(inplace=True)
    (conv2): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (shortcut): Conv1d(64, 64, ke

In [14]:
!cd "/content/drive/MyDrive/CUHK_Stuff/AI-for-Business-Research/Assignment/Final Project/Study2/code" && python 04_baselines.py

LOADING DATA
  Train (SMOTE): (201136, 47)
  Val:  (22500, 47)
  Test: (22501, 47)
  Features: 47

1. LOGISTIC REGRESSION
  Optimal threshold (from val): 0.06 | Val F1: 0.1229
  Test F1=0.1260 | AUC-ROC=0.6301 | Time=1.4s

2. RANDOM FOREST
  Optimal threshold (from val): 0.14 | Val F1: 0.1211
  Test F1=0.1217 | AUC-ROC=0.6210 | Time=3.1s

3. XGBOOST
  Optimal threshold (from val): 0.08 | Val F1: 0.1246
  Test F1=0.1206 | AUC-ROC=0.6347 | Time=0.9s

4. MLP (PyTorch)
    Epoch   1 | train_loss=0.1483 | val_loss=0.1718
    Epoch  10 | train_loss=0.0947 | val_loss=0.1712
    Epoch  20 | train_loss=0.0940 | val_loss=0.1710
    Epoch  30 | train_loss=0.0910 | val_loss=0.1710
    Epoch  40 | train_loss=0.0895 | val_loss=0.1715
    Epoch  50 | train_loss=0.0889 | val_loss=0.1715
  Optimal threshold (from val): 0.05 | Val F1: 0.1253
  Test F1=0.1193 | AUC-ROC=0.6375 | Time=81.5s

SAVING BASELINE RESULTS
  Saved: /content/drive/.shortcut-targets-by-id/1KKOoZevY1ahDrKYLLQ-dQDM6ItZ8AsDk/Final Proj

In [15]:
!cd "/content/drive/MyDrive/CUHK_Stuff/AI-for-Business-Research/Assignment/Final Project/Study2/code" && python 05_train_evaluate.py

LOADING DATA
  Features: 47
  Train (SMOTE): 201,136 | Val: 22,500 | Test: 22,501

TRAINING 1D-CNN WITH RESIDUAL + SELF-ATTENTION
  Model parameters: 44,242
  Training on cuda | epochs=30 | lr=0.001 | patience=7
  Train batches: 786 | Val batches: 88
    Epoch   1/30 | train_loss=0.1342 acc=0.9637 | val_loss=0.1788 acc=0.9573 | lr=0.001000
    Epoch   5/30 | train_loss=0.0992 acc=0.9746 | val_loss=0.1784 acc=0.9565 | lr=0.001000
    Epoch  10/30 | train_loss=0.0963 acc=0.9753 | val_loss=0.1771 acc=0.9578 | lr=0.001000
    Epoch  15/30 | train_loss=0.0943 acc=0.9757 | val_loss=0.1710 acc=0.9578 | lr=0.000500
    Epoch  20/30 | train_loss=0.0933 acc=0.9759 | val_loss=0.1715 acc=0.9577 | lr=0.000500
    Epoch  25/30 | train_loss=0.0929 acc=0.9760 | val_loss=0.1723 acc=0.9574 | lr=0.000500
    Epoch  30/30 | train_loss=0.0914 acc=0.9763 | val_loss=0.1747 acc=0.9563 | lr=0.000250
  Early stopping at epoch 30 (no improvement for 7 epochs)
  Best model saved to: /content/drive/.shortcut-targe